# 08 - Export Power BI

**Unic objectiu d'aquest notebook:** ser el pont entre Python i Power BI. No fa ETL, no
recalcula res, no crea visualitzacions ni mapes. Nomes llegeix els CSV finals ja generats,
els valida, uniformitza noms de columna i els exporta.

## Datasets carregats (els 16 finals del projecte, cap intermedi)

**Taules d'entitat/serie (notebooks 03-05):**

| Dataset | Notebook | Contingut |
|---|---|---|
| `population_final.csv` | 03_merge_population | Poblacio per assentament i any (1993-2025) |
| `settlements_points.csv` | 05_etl_geodata | Assentaments geolocalitzats |
| `outposts_points.csv` | 05_etl_geodata | Outposts geolocalitzats |
| `demolitions_wb.csv` | 04_etl_violence | Demolicions d'habitatges (2006-2026) |
| `fatalities_settlers_wb.csv` | 04_etl_violence | Victimes mortals a mans de colons (2000-2026) |
| `wb_daily_post7o.csv` | 04_etl_violence | Serie mensual post-7-O |
| `wb_daily_post7o_daily.csv` | 04_etl_violence | Serie diaria post-7-O |

**Taules resum (notebook 06, ja pensades per a Power BI):**

| Dataset | Notebook | Contingut |
|---|---|---|
| `population_annual_summary.csv` | 06_analysis_temporal | Poblacio total i creixement interanual, per any |
| `demolitions_annual_summary.csv` | 06_analysis_temporal | Demolicions, unitats i desplaçats, per any |
| `demolitions_district_summary.csv` | 06_analysis_temporal | Demolicions totals per districte |
| `fatalities_annual_summary.csv` | 06_analysis_temporal | Victimes mortals per any |
| `fatalities_district_summary.csv` | 06_analysis_temporal | Victimes totals per districte |
| `outposts_annual_summary.csv` | 06_analysis_temporal | Nous outposts i acumulat, per any |
| `settlements_vs_outposts_founding.csv` | 06_analysis_temporal | Fundacio formal (assentaments) vs. informal (outposts), per any |
| `post7o_period_comparison.csv` | 06_analysis_temporal | Ritmes mensuals pre/post 7-O 2023 |
| `master_annual_summary.csv` | 06_analysis_temporal | Taula fet anual unica (poblacio+demolicions+victimes+outposts) |

**Deliberadament exclosos** (versions intermedies del notebook 01, ja superades):
`peacenow_population_long.csv`, `peacenow_settlements_geo.csv`, `peacenow_outposts_geo.csv`.

## Outputs
- `outputs/powerbi/*.csv` - 16 CSV definitius, noms de columna uniformitzats
- `outputs/powerbi_data_dictionary.md` - diccionari de dades + model relacional recomanat


## 1. Importacio de llibreries

In [1]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Configuracio: registre de datasets finals

Un unic diccionari controla tot el notebook. Afegir un dataset nou nomes requereix afegir
una entrada aqui.


In [2]:
CLEAN = "data/clean"
OUT_DIR = "outputs/powerbi"
DICT_PATH = "outputs/powerbi_data_dictionary.md"

os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = {
    "population_final": {
        "path": f"{CLEAN}/population_final.csv",
        "source_notebook": "03_merge_population",
        "description": "Poblacio dels assentaments israelians a Cisjordania, per assentament i any (1993-2025). Fusio de Peace Now i JVL.",
        "grain": "1 fila = 1 assentament x 1 any",
    },
    "settlements_points": {
        "path": f"{CLEAN}/settlements_points.csv",
        "source_notebook": "05_etl_geodata",
        "description": "Assentaments israelians geolocalitzats, amb atributs descriptius (tipologia, any de fundacio, distancia a la Green Line).",
        "grain": "1 fila = 1 assentament",
    },
    "outposts_points": {
        "path": f"{CLEAN}/outposts_points.csv",
        "source_notebook": "05_etl_geodata",
        "description": "Outposts (avantpostos no autoritzats) geolocalitzats, amb atributs descriptius.",
        "grain": "1 fila = 1 outpost",
    },
    "demolitions_wb": {
        "path": f"{CLEAN}/demolitions_wb.csv",
        "source_notebook": "04_etl_violence",
        "description": "Demolicions d'habitatges i estructures palestines a Cisjordania (B'Tselem, 2006-2026).",
        "grain": "1 fila = 1 esdeveniment de demolicio",
    },
    "fatalities_settlers_wb": {
        "path": f"{CLEAN}/fatalities_settlers_wb.csv",
        "source_notebook": "04_etl_violence",
        "description": "Palestins morts a mans de colons a Cisjordania (B'Tselem, 2000-2026).",
        "grain": "1 fila = 1 victima",
    },
    "wb_daily_post7o": {
        "path": f"{CLEAN}/wb_daily_post7o.csv",
        "source_notebook": "04_etl_violence",
        "description": "Serie mensual agregada post-7 d'octubre de 2023: morts, ferits i atacs de colons.",
        "grain": "1 fila = 1 mes",
    },
    "wb_daily_post7o_daily": {
        "path": f"{CLEAN}/wb_daily_post7o_daily.csv",
        "source_notebook": "04_etl_violence",
        "description": "Serie diaria post-7 d'octubre de 2023: acumulats de morts, ferits i atacs de colons.",
        "grain": "1 fila = 1 dia",
    },
    "population_annual_summary": {
        "path": f"{CLEAN}/population_annual_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Poblacio total dels assentaments i creixement interanual, per any.",
        "grain": "1 fila = 1 any",
    },
    "demolitions_annual_summary": {
        "path": f"{CLEAN}/demolitions_annual_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Demolicions, unitats d'habitatge i persones desplaçades, agregat per any.",
        "grain": "1 fila = 1 any",
    },
    "demolitions_district_summary": {
        "path": f"{CLEAN}/demolitions_district_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Demolicions totals, unitats i persones desplaçades, agregat per districte.",
        "grain": "1 fila = 1 districte",
    },
    "fatalities_annual_summary": {
        "path": f"{CLEAN}/fatalities_annual_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Victimes mortals a mans de colons, agregat per any (anys sense victimes inclosos amb 0).",
        "grain": "1 fila = 1 any",
    },
    "fatalities_district_summary": {
        "path": f"{CLEAN}/fatalities_district_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Victimes mortals totals, agregat per districte.",
        "grain": "1 fila = 1 districte",
    },
    "outposts_annual_summary": {
        "path": f"{CLEAN}/outposts_annual_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Nous outposts fundats i acumulat, per any.",
        "grain": "1 fila = 1 any",
    },
    "settlements_vs_outposts_founding": {
        "path": f"{CLEAN}/settlements_vs_outposts_founding.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Comparativa de fundacio formal (assentaments) vs. informal (outposts), per any.",
        "grain": "1 fila = 1 any",
    },
    "post7o_period_comparison": {
        "path": f"{CLEAN}/post7o_period_comparison.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Ritmes mensuals abans/despres del 7-O 2023 per a demolicions, victimes i outposts, purament descriptiu.",
        "grain": "1 fila = 1 metrica",
    },
    "master_annual_summary": {
        "path": f"{CLEAN}/master_annual_summary.csv",
        "source_notebook": "06_analysis_temporal",
        "description": "Taula fet anual unica: poblacio + demolicions + victimes + outposts, per any. Pensada com a taula central per a Power BI.",
        "grain": "1 fila = 1 any",
    },
}

print(f"{len(DATASETS)} datasets registrats")

16 datasets registrats


## 3. Carrega

In [3]:
dfs = {}
for name, meta in DATASETS.items():
    dfs[name] = pd.read_csv(meta["path"])
    print(f"{name:28s} {dfs[name].shape[0]:>6} files x {dfs[name].shape[1]:>2} columnes")

FileNotFoundError: [Errno 2] No such file or directory: 'data/clean/population_final.csv'

---
## 4. Validacio final

Per a cada dataset: files/columnes, tipus de dades, nuls, duplicats, i una comprovacio
basica de coherencia (dates parsejables, rangs de valors versemblants). No es corregeix
res aqui, nomes es documenta.


In [ ]:
def quality_check(name, df):
    n_dup = df.duplicated().sum()

    # Deteccio de columnes de data per nom (no es converteixen, nomes es validen)
    date_cols = [c for c in df.columns if "date" in c.lower()]
    unparseable_dates = {}
    for c in date_cols:
        parsed = pd.to_datetime(df[c], errors="coerce")
        n_bad = parsed.isna().sum() - df[c].isna().sum()
        if n_bad > 0:
            unparseable_dates[c] = int(n_bad)

    null_cells = df.isna().sum().sum()
    total_cells = df.shape[0] * df.shape[1]

    return {
        "dataset": name,
        "n_rows": df.shape[0],
        "n_cols": df.shape[1],
        "n_duplicate_rows": int(n_dup),
        "cols_with_nulls": int((df.isna().sum() > 0).sum()),
        "pct_null_cells": round(100 * null_cells / total_cells, 2),
        "date_cols_checked": len(date_cols),
        "unparseable_dates": unparseable_dates if unparseable_dates else "-",
    }

quality_rows = [quality_check(name, df) for name, df in dfs.items()]
quality_summary = pd.DataFrame(quality_rows)
quality_summary

,dataset,n_rows,n_cols,n_duplicate_rows,cols_with_nulls,pct_null_cells,date_cols_checked,unparseable_dates
0,population_final,4008,4,0,0,0.00,0,-
1,settlements_points,147,11,0,4,3.22,0,-
2,outposts_points,383,11,0,4,2.33,0,-
3,demolitions_wb,5465,10,1333,2,0.01,1,-
4,fatalities_settlers_wb,118,13,0,2,2.93,2,-
5,wb_daily_post7o,34,10,0,0,0.00,1,-
6,wb_daily_post7o_daily,1012,10,0,2,12.17,1,-
7,population_annual_summary,33,4,0,1,0.76,0,-
8,demolitions_annual_summary,21,5,0,0,0.00,0,-
9,demolitions_district_summary,13,4,0,0,0.00,0,-


**Troballes a documentar (no es corregeixen, nomes es reporten):**

- **`demolitions_wb`: 1.333 files exactament duplicades (de 5.465).** No hi ha cap columna
  d'identificador unic d'esdeveniment a la font (B'Tselem), per tant no es pot distingir
  entre "el mateix registre repetit per error" i "dos esdeveniments reals identics en tots
  els camps disponibles" (p. ex. dues estructures demolides el mateix dia, a la mateixa
  localitat, amb les mateixes unitats). Es deixen tal qual — eliminar-les suposaria un
  recalcul de xifres, fora d'abast d'aquest notebook.
- **`demolitions_wb.structure_type`** conte el valor `"residental"` (error tipografic de la
  font, hauria de ser `"residential"`). No es corregeix aqui — nomes es documenta.
- La resta de datasets no presenten duplicats ni dates no parsejables.


In [ ]:
# Verificacio addicional de rangs versemblants (nomes lectura, no es modifica res)
checks = []

checks.append(("settlements_points: lat/lon dins de Cisjordania (31-33 / 34-36)",
                dfs["settlements_points"]["lat"].between(31, 33).all() and
                dfs["settlements_points"]["lon"].between(34, 36).all()))

checks.append(("outposts_points: lat/lon dins de Cisjordania (31-33 / 34-36)",
                dfs["outposts_points"]["lat"].between(31, 33).all() and
                dfs["outposts_points"]["lon"].between(34, 36).all()))

checks.append(("demolitions_wb: cap valor negatiu en housing_units/people_homeless",
                (dfs["demolitions_wb"][["housing_units", "people_homeless", "minors_homeless"]] >= 0).all().all()))

checks.append(("fatalities_settlers_wb: Age dins d'un rang versemblant (0-110)",
                dfs["fatalities_settlers_wb"]["Age"].dropna().between(0, 110).all()))

checks.append(("population_final: population >= 0",
                (dfs["population_final"]["population"] >= 0).all()))

checks.append(("*_annual_summary: anys dins del rang esperat (1967-2026)",
                all(
                    dfs[t]["year"].between(1967, 2026).all()
                    for t in ["population_annual_summary", "demolitions_annual_summary",
                              "fatalities_annual_summary", "outposts_annual_summary",
                              "settlements_vs_outposts_founding", "master_annual_summary"]
                )))

checks.append(("demolitions_district_summary / fatalities_district_summary: vocabulari de districte compartit",
                len(set(dfs["demolitions_district_summary"]["district"]) & set(dfs["fatalities_district_summary"]["district"])) >= 10))

for label, ok in checks:
    print(f"{'OK ' if ok else 'FALLA '} {label}")

OK  settlements_points: lat/lon dins de Cisjordania (31-33 / 34-36)
OK  outposts_points: lat/lon dins de Cisjordania (31-33 / 34-36)
OK  demolitions_wb: cap valor negatiu en housing_units/people_homeless
OK  fatalities_settlers_wb: Age dins d'un rang versemblant (0-110)
OK  population_final: population >= 0
OK  *_annual_summary: anys dins del rang esperat (1967-2026)
OK  demolitions_district_summary / fatalities_district_summary: vocabulari de districte compartit


**Troballa addicional:** `demolitions_district_summary.district` i
`fatalities_district_summary.district` fan servir el mateix vocabulari de districte (11 de
13 valors coincideixen exactament — `demolitions_district_summary` inclou a mes `Jenin` i
`Israel`, absents a l'altra taula). A diferencia dels noms de localitat de
`demolitions_wb`/`fatalities_settlers_wb` (que no tenen equivalent a les dimensions
d'assentaments), aquest camp `district` si que es una clau compartida real entre les dues
taules resum — es documenta com a relacio recomanada a la Seccio 8.


---
## 5. Uniformitzacio de noms de columna

Funcio generica de `snake_case` (minuscules, sense accents, sense espais ni caracters
especials) aplicada a totes les taules per igual, perque els noms siguin consistents entre
datasets. Nomes es toca la capcalera — cap valor de dades es modifica.


In [ ]:
def to_snake_case(col: str) -> str:
    col = str(col).strip()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = re.sub(r"[^0-9a-zA-Z]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.lower()


rename_log = {}
for name, df in dfs.items():
    original_cols = list(df.columns)
    new_cols = [to_snake_case(c) for c in original_cols]
    changed = {old: new for old, new in zip(original_cols, new_cols) if old != new}
    if changed:
        rename_log[name] = changed
    df.columns = new_cols

print("Columnes renombrades per dataset:")
for name, changes in rename_log.items():
    print(f"\n{name}:")
    for old, new in changes.items():
        print(f"  {old!r} -> {new!r}")
if not rename_log:
    print("Cap canvi necessari - tots els noms ja eren consistents.")

Columnes renombrades per dataset:

fatalities_settlers_wb:
  'Name' -> 'name'
  'Age' -> 'age'
  'Gender' -> 'gender'
  'Ammunition' -> 'ammunition'
  'Notes' -> 'notes'


---
## 6. Exportacio a `outputs/powerbi/`


In [ ]:
for name, df in dfs.items():
    out_path = f"{OUT_DIR}/{name}.csv"
    df.to_csv(out_path, index=False)
    print(f"OK {out_path} ({df.shape[0]} files, {df.shape[1]} columnes)")

OK outputs/powerbi/population_final.csv (4008 files, 4 columnes)
OK outputs/powerbi/settlements_points.csv (147 files, 11 columnes)
OK outputs/powerbi/outposts_points.csv (383 files, 11 columnes)
OK outputs/powerbi/demolitions_wb.csv (5465 files, 10 columnes)
OK outputs/powerbi/fatalities_settlers_wb.csv (118 files, 13 columnes)
OK outputs/powerbi/wb_daily_post7o.csv (34 files, 10 columnes)


OK outputs/powerbi/wb_daily_post7o_daily.csv (1012 files, 10 columnes)
OK outputs/powerbi/population_annual_summary.csv (33 files, 4 columnes)
OK outputs/powerbi/demolitions_annual_summary.csv (21 files, 5 columnes)
OK outputs/powerbi/demolitions_district_summary.csv (13 files, 4 columnes)
OK outputs/powerbi/fatalities_annual_summary.csv (27 files, 2 columnes)
OK outputs/powerbi/fatalities_district_summary.csv (11 files, 2 columnes)
OK outputs/powerbi/outposts_annual_summary.csv (31 files, 3 columnes)
OK outputs/powerbi/settlements_vs_outposts_founding.csv (56 files, 3 columnes)
OK outputs/powerbi/post7o_period_comparison.csv (3 files, 4 columnes)
OK outputs/powerbi/master_annual_summary.csv (34 files, 9 columnes)


---
## 7. Diccionari de dades (`powerbi_data_dictionary.md`)

Generat automaticament a partir dels `dtypes` i estadistiques reals de cada taula ja
exportada — no s'escriu cap descripcio de columna a ma que pugui desincronitzar-se de les
dades.


In [ ]:
PRIMARY_KEYS = {
    "population_final": "settlement + year (composta)",
    "settlements_points": "name (unic; db_id tambe unic i pot fer-ne de PK tecnica)",
    "outposts_points": "name (unic; db_id NO es unic ni complet - 33 valors nuls)",
    "demolitions_wb": "Cap - no hi ha identificador unic d'esdeveniment a la font (veure Seccio 4)",
    "fatalities_settlers_wb": "name (unic en aquest dataset; name + date_event com a alternativa mes robusta)",
    "wb_daily_post7o": "year + month (composta)",
    "wb_daily_post7o_daily": "date",
    "population_annual_summary": "year",
    "demolitions_annual_summary": "year",
    "demolitions_district_summary": "district",
    "fatalities_annual_summary": "year",
    "fatalities_district_summary": "district",
    "outposts_annual_summary": "year",
    "settlements_vs_outposts_founding": "year",
    "post7o_period_comparison": "metric",
    "master_annual_summary": "year",
}

FOREIGN_KEYS = {
    "population_final": "settlement -> settlements_points.name (relacio aproximada: ~86% de coincidencia per a l'any 2024; els noms de la font JVL de 2025 no coincideixen be amb Peace Now, veure notebook 07)",
    "settlements_points": "Cap",
    "outposts_points": "Cap",
    "demolitions_wb": "Cap - locality/district son noms de localitats palestines, sense cap taula de dimensio local amb aquesta nomenclatura (veure notebook 07)",
    "fatalities_settlers_wb": "Cap - mateixa limitacio que demolitions_wb",
    "wb_daily_post7o": "Cap FK directa; year+month es correspon amb el rang de dates de wb_daily_post7o_daily (mateixa font, granularitat diferent)",
    "wb_daily_post7o_daily": "Cap FK directa; es la font de la qual es deriva wb_daily_post7o (agregacio mensual)",
    "population_annual_summary": "year -> cap taula de calendari propia (recomanat crear-ne una a Power BI, veure Seccio 8). Redundant amb master_annual_summary (mateixes columnes de poblacio)",
    "demolitions_annual_summary": "year -> mateixa nota que population_annual_summary. Redundant amb master_annual_summary",
    "demolitions_district_summary": "district -> fatalities_district_summary.district (vocabulari compartit, veure nota Seccio 4)",
    "fatalities_annual_summary": "year -> mateixa nota que population_annual_summary. Redundant amb master_annual_summary",
    "fatalities_district_summary": "district -> demolitions_district_summary.district",
    "outposts_annual_summary": "year -> mateixa nota que population_annual_summary. Redundant amb master_annual_summary",
    "settlements_vs_outposts_founding": "year -> mateixa nota que population_annual_summary",
    "post7o_period_comparison": "Cap - taula de nomes 3 files, us com a taula de referencia autonoma (targetes KPI a Power BI)",
    "master_annual_summary": "year -> taula de calendari recomanada. Aquesta taula ja incorpora el contingut de population_annual_summary/demolitions_annual_summary/fatalities_annual_summary/outposts_annual_summary",
}

def dtype_to_powerbi(dtype):
    s = str(dtype)
    if "int" in s:
        return "Enter (Integer)"
    if "float" in s:
        return "Decimal"
    if "bool" in s:
        return "Booleà"
    return "Text"

lines = []
lines.append("# Diccionari de dades - Power BI\n")
lines.append("Generat automaticament pel notebook `08_export_powerbi.ipynb`. ")
lines.append("Font unica de veritat: no editar aquest fitxer a ma, regenerar-lo des del notebook.\n")

for name, df in dfs.items():
    meta = DATASETS[name]
    lines.append(f"## `{name}.csv`\n")
    lines.append(f"{meta['description']}\n")
    lines.append(f"- **Notebook origen:** {meta['source_notebook']}")
    lines.append(f"- **Granularitat:** {meta['grain']}")
    lines.append(f"- **Registres:** {df.shape[0]:,}")
    lines.append(f"- **Columnes:** {df.shape[1]}")
    lines.append(f"- **Clau primaria:** {PRIMARY_KEYS.get(name, 'n/d')}")
    lines.append(f"- **Claus foranes / relacions:** {FOREIGN_KEYS.get(name, 'n/d')}\n")
    lines.append("| Columna | Tipus (pandas) | Tipus recomanat a Power BI | % nuls |")
    lines.append("|---|---|---|---|")
    for col in df.columns:
        pct_null = round(100 * df[col].isna().mean(), 1)
        lines.append(f"| `{col}` | {df[col].dtype} | {dtype_to_powerbi(df[col].dtype)} | {pct_null}% |")
    lines.append("")

data_dict_md = "\n".join(lines)
print(data_dict_md[:1500])
print("...")

# Diccionari de dades - Power BI

Generat automaticament pel notebook `08_export_powerbi.ipynb`. 
Font unica de veritat: no editar aquest fitxer a ma, regenerar-lo des del notebook.

## `population_final.csv`

Poblacio dels assentaments israelians a Cisjordania, per assentament i any (1993-2025). Fusio de Peace Now i JVL.

- **Notebook origen:** 03_merge_population
- **Granularitat:** 1 fila = 1 assentament x 1 any
- **Registres:** 4,008
- **Columnes:** 4
- **Clau primaria:** settlement + year (composta)
- **Claus foranes / relacions:** settlement -> settlements_points.name (relacio aproximada: ~86% de coincidencia per a l'any 2024; els noms de la font JVL de 2025 no coincideixen be amb Peace Now, veure notebook 07)

| Columna | Tipus (pandas) | Tipus recomanat a Power BI | % nuls |
|---|---|---|---|
| `settlement` | str | Text | 0.0% |
| `year` | int64 | Enter (Integer) | 0.0% |
| `population` | int64 | Enter (Integer) | 0.0% |
| `source` | str | Text | 0.0% |

## `settlements_points.

---
## 8. Model relacional recomanat per a Power BI

**No es crea cap fitxer .pbix aqui — nomes es documenta la proposta.**

### Taula de fets principal
Cap taula unica cobreix tot el projecte (les fonts de violencia i les d'assentaments fan
servir vocabularis geografics diferents, sense clau comuna — ja documentat al notebook 07).
El model recomanat es de **fets multiples**, un per fenomen i granularitat:

- **`wb_daily_post7o_daily`** (grain=dia) — la serie temporal mes granular i completa.
- **`demolitions_wb`** (grain=esdeveniment) — fet d'impacte territorial detallat.
- **`fatalities_settlers_wb`** (grain=esdeveniment) — fet de victimes detallat.
- **`master_annual_summary`** (grain=any) — **recomanat com a fet principal per a
  targetes/KPI i grafics anuals a nivell de projecte**, perque ja consolida poblacio,
  demolicions, victimes i outposts en una sola fila per any.

### Taules "resum" redundants amb `master_annual_summary`
`population_annual_summary`, `demolitions_annual_summary`, `fatalities_annual_summary` i
`outposts_annual_summary` contenen (en columnes soltes) el mateix que ja hi ha a
`master_annual_summary`. No se n'ha eliminat cap — es documenten totes — pero a Power BI
**nomes cal carregar-hi `master_annual_summary` per a l'us habitual**; les individuals
serveixen com a backup/detall (p. ex. si en algun moment cal una columna que no es va
incloure a la fusio, com `minors_homeless`, present a `demolitions_annual_summary` pero no
a `master_annual_summary`).

### Taules de dimensio
- **`settlements_points`** (dim_settlements) — 1 fila = 1 assentament.
- **`outposts_points`** (dim_outposts) — 1 fila = 1 outpost.
- **`demolitions_district_summary`** i **`fatalities_district_summary`** — es poden fer
  servir com a base d'una **dim_district** (11 de 13 valors coincideixen entre totes dues,
  veure Seccio 4). No es crea aqui una taula de dimensio unificada per no fer ETL
  addicional — es documenta la possibilitat.

### Relacions recomanades

| Des de | Cap a | Cardinalitat | Direccio del filtre | Nota |
|---|---|---|---|---|
| `population_final.settlement` | `settlements_points.name` | Molts-a-1 | Unica (dim -> fet) | Coincidencia ~86% per a 2024; NO activar "assumir integritat referencial" |
| `wb_daily_post7o.[year,month]` | `wb_daily_post7o_daily.[year,month]` | 1-a-molts | No relacionar directament | Mateixa font a dues granularitats — usar nomes la diaria i agregar amb una taula de Dates |
| `demolitions_district_summary.district` | `fatalities_district_summary.district` | 1-a-1 (aprox.) | Bidireccional opcional | Vocabulari compartit en 11/13 valors; util per a un grafic combinat per districte |
| `demolitions_wb.district` | `demolitions_district_summary.district` | Molts-a-1 | Unica (dim -> fet) | Mateixa font, granularitats diferents |
| `fatalities_settlers_wb.district` | `fatalities_district_summary.district` | Molts-a-1 | Unica (dim -> fet) | Mateixa font, granularitats diferents |
| `demolitions_wb` / `fatalities_settlers_wb` (locality) | *(cap dimensio)* | - | - | Sense clau comuna amb `settlements_points`/`outposts_points` (localitats palestines vs. consells regionals israelians) |

### Recomanacio addicional (nomes per documentar, no s'implementa aqui)
Crear una **taula de Dates nativa a Power BI** (`Nova taula > CALENDAR(...)`) que cobreixi
tot el rang 1967-2026, i relacionar-hi `year` de totes les taules anuals
(`master_annual_summary`, `population_annual_summary`, `demolitions_annual_summary`,
`fatalities_annual_summary`, `outposts_annual_summary`, `settlements_vs_outposts_founding`)
mes `date`/`year` de `population_final`, `demolitions_wb`, `fatalities_settlers_wb` i
`wb_daily_post7o_daily`. Totes com a molts-a-1, direccio unica cap als fets. Aixo permet
una segmentacio temporal creuada entre totes les taules sense necessitat de cap clau
geografica compartida.


In [ ]:
with open(DICT_PATH, "w", encoding="utf-8") as f:
    f.write(data_dict_md)
print(f"OK {DICT_PATH}")

OK outputs/powerbi_data_dictionary.md


---
## 9. Resum per a la memoria

Taula final: quins datasets formen el lliurable, quin notebook els genera, i quin us se'n
fa a la resta del projecte.


In [ ]:
summary_rows = []
for name, meta in DATASETS.items():
    summary_rows.append({
        "dataset": f"{name}.csv",
        "notebook_origen": meta["source_notebook"],
        "us_al_projecte": {
            "population_final": "Power BI (fet demografic) + notebook 07 (mida dels marcadors)",
            "settlements_points": "Power BI (dimensio) + notebook 07 (mapa)",
            "outposts_points": "Power BI (dimensio) + notebook 07 (mapa)",
            "demolitions_wb": "Power BI (fet) + informe final",
            "fatalities_settlers_wb": "Power BI (fet) + informe final",
            "wb_daily_post7o": "Power BI (fet, granularitat mensual) + informe final",
            "wb_daily_post7o_daily": "Power BI (fet, granularitat diaria) + notebook 07 (panell KPI)",
            "population_annual_summary": "Power BI (detall/backup, veure Seccio 8) + informe final",
            "demolitions_annual_summary": "Power BI (detall/backup, veure Seccio 8) + informe final",
            "demolitions_district_summary": "Power BI (dim_district candidata) + informe final",
            "fatalities_annual_summary": "Power BI (detall/backup, veure Seccio 8) + informe final",
            "fatalities_district_summary": "Power BI (dim_district candidata) + informe final",
            "outposts_annual_summary": "Power BI (detall/backup, veure Seccio 8) + informe final",
            "settlements_vs_outposts_founding": "Power BI (grafic comparatiu) + informe final",
            "post7o_period_comparison": "Power BI (targetes KPI) + informe final",
            "master_annual_summary": "Power BI (fet principal anual) + informe final",
        }.get(name, "n/d"),
    })

final_summary = pd.DataFrame(summary_rows)
final_summary

,dataset,notebook_origen,us_al_projecte
0,population_final.csv,03_merge_population,Power BI (fet demografic) + notebook 07 (mida ...
1,settlements_points.csv,05_etl_geodata,Power BI (dimensio) + notebook 07 (mapa)
2,outposts_points.csv,05_etl_geodata,Power BI (dimensio) + notebook 07 (mapa)
3,demolitions_wb.csv,04_etl_violence,Power BI (fet) + informe final
4,fatalities_settlers_wb.csv,04_etl_violence,Power BI (fet) + informe final
5,wb_daily_post7o.csv,04_etl_violence,"Power BI (fet, granularitat mensual) + informe..."
6,wb_daily_post7o_daily.csv,04_etl_violence,"Power BI (fet, granularitat diaria) + notebook..."
7,population_annual_summary.csv,06_analysis_temporal,"Power BI (detall/backup, veure Seccio 8) + inf..."
8,demolitions_annual_summary.csv,06_analysis_temporal,"Power BI (detall/backup, veure Seccio 8) + inf..."
9,demolitions_district_summary.csv,06_analysis_temporal,Power BI (dim_district candidata) + informe final


In [ ]:
print("Lliurable 08 completat.")
print(f"  {len(DATASETS)} CSV exportats a {OUT_DIR}/")
print(f"  Diccionari de dades: {DICT_PATH}")
print(f"  Cap dada modificada - nomes capcaleres uniformitzades")

Lliurable 08 completat.
  16 CSV exportats a outputs/powerbi/
  Diccionari de dades: outputs/powerbi_data_dictionary.md
  Cap dada modificada - nomes capcaleres uniformitzades
